# STEP 3. 라벨 생성 + 표본 필터 + 로그 변환

## 라벨 정의

```
순증감(t) = 개업_점포_수(t) − 폐업_점포_수(t)
y(t)      = 1 if 순증감(t+1) < 0 else 0     ← 다음 분기 순감소 전환
```

**주의 두 가지**

- `shift(-1)` 은 반드시 (상권, 업종) 그룹 내에서 **분기 오름차순 정렬 후** 수행합니다. 안 그러면 A상권 마지막 분기 라벨에 B상권 첫 분기 값이 들어갑니다
- 관측 분기가 연속하지 않으면 라벨을 만들지 않습니다. `shift(-1)` 은 "행 순서상 다음"을 가져올 뿐 "분기상 다음"을 보장하지 않습니다

## 20261 의 역할

13분기를 적재했지만 **20261 은 라벨 공급용**입니다. 20254 의 `y` 를 만들려면 20261 의 개업·폐업이 필요합니다. 20261 자체는 다음 분기가 없어 라벨을 못 만들고 여기서 탈락합니다.

→ 학습 구간 **20231 ~ 20254, 12분기**

## 필터 근거

| 필터 | 이유 |
|---|---|
| 라벨 존재 | y 없으면 학습 불가 |
| `전체_점포_수` ≥ 5 | 점포 3~4개 칸은 개업 1건·폐업 1건만으로 라벨이 뒤집힘. 신호보다 노이즈가 큼 |
| 유동인구 존재 | H2 매칭 공변량 ② 없으면 매칭 불가 |
| 면적 > 0 | 밀도 계산 불가능한 이상치 |

> ⚠️ **이번 개편의 핵심 변화** — 5개 이상 필터가 `일반_점포_수`(비프랜차이즈)가 아니라 `전체_점포_수`(총수) 기준이 됐습니다. 통과하는 칸이 늘어나고, **프랜차이즈 밀집 업종이 특히 많이 들어옵니다.** 4-2 에서 그 변화를 수치로 확인합니다.


In [1]:
import sys, os
from pathlib import Path

# 노트북이 어디서 열리든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("프로젝트 루트:", ROOT)

import importlib
if "config" in sys.modules:
    importlib.reload(sys.modules["config"])

프로젝트 루트: c:\Users\spide\ai-data-bootcamp\project\h2_nb


In [2]:
from config import (PROC, MIN_STORES, QUARTERS, COVARIATES,
                    TRAIN_QUARTERS, VALID_Q, SCORE_Q)

df = pd.read_pickle(PROC / "02_merged.pkl")
assert "전체_점포_수" in df.columns, "STEP 2 를 새 버전으로 다시 실행하세요"

def quarter_index(code):
    """20241 → 연속 정수. 분기 연속성 판정용"""
    return (code // 10) * 4 + (code % 10 - 1)

df["분기idx"]    = quarter_index(df["기준_년분기_코드"])
df["순증감"]      = df["개업_점포_수"] - df["폐업_점포_수"]
df["순감소_현재"] = (df["순증감"] < 0).astype(int)

print(f"입력 {df.shape}")
print(f"분기 {len(QUARTERS)}개 (마지막 {QUARTERS[-1]} 은 라벨 공급용)")

입력 (160352, 29)
분기 13개 (마지막 20261 은 라벨 공급용)


## 3-1. 라벨 생성

In [3]:
df = df.sort_values(["상권_코드", "서비스_업종_코드", "분기idx"]).reset_index(drop=True)
g = df.groupby(["상권_코드", "서비스_업종_코드"], sort=False)

df["y"]          = g["순감소_현재"].shift(-1)
df["다음분기idx"] = g["분기idx"].shift(-1)

연속 = (df["다음분기idx"] - df["분기idx"]) == 1
끊김 = df["y"].notna() & ~연속
df.loc[~연속, "y"] = np.nan

print(f"라벨 생성 {df['y'].notna().sum():,} / 전체 {len(df):,}")
print(f"  └ 분기 불연속으로 무효화한 라벨 {끊김.sum():,}건\n")
display(df.groupby("기준_년분기_코드")["y"].agg(행수="size", 라벨있음="count")
        .assign(라벨률=lambda t: (t["라벨있음"]/t["행수"]).round(3)))
print(f"※ {QUARTERS[-1]} 의 라벨률 0 은 정상입니다 (다음 분기가 없음)")

라벨 생성 147,138 / 전체 160,352
  └ 분기 불연속으로 무효화한 라벨 140건



,행수,라벨있음,라벨률
기준_년분기_코드,,,
20231,12397,12315,0.993
20232,12409,12310,0.992
20233,12393,12312,0.993
20234,12405,12334,0.994
20241,12418,12325,0.993
20242,12388,12296,0.993
20243,12348,12263,0.993
20244,12320,12245,0.994
20251,12304,12236,0.994


※ 20261 의 라벨률 0 은 정상입니다 (다음 분기가 없음)


## 3-2. 표본 필터 — 단계별 손실 추적

In [4]:
steps = [("원본", df)]
d = df[df["y"].notna()].copy();               steps.append(("라벨 존재", d))
d = d[d["전체_점포_수"] >= MIN_STORES];        steps.append((f"점포 {MIN_STORES}개↑", d))
d = d[d["총_유동인구_수"].notna()];             steps.append(("유동인구 존재", d))
d = d[d["영역_면적"] > 0];                     steps.append(("면적 유효", d))

prev = None
for nm, s in steps:
    drop = "" if prev is None else f"  (-{prev-len(s):,})"
    print(f"  {nm:14} {len(s):>8,}{drop}")
    prev = len(s)

d["y"] = d["y"].astype(int)
print(f"\n최종 표본 {len(d):,}행 | 양성률(순감소) {d['y'].mean():.1%}")
print(f"학습 구간 {d['기준_년분기_코드'].min()} ~ {d['기준_년분기_코드'].max()} "
      f"({d['기준_년분기_코드'].nunique()}분기)")
print(f"무사건 칸(개업=0 & 폐업=0) 비율 : "
      f"{((d['개업_점포_수']==0)&(d['폐업_점포_수']==0)).mean():.1%}")

  원본              160,352
  라벨 존재           147,138  (-13,214)
  점포 5개↑           67,495  (-79,643)
  유동인구 존재          67,475  (-20)
  면적 유효            67,475  (-0)

최종 표본 67,475행 | 양성률(순감소) 25.5%
학습 구간 20231 ~ 20254 (12분기)
무사건 칸(개업=0 & 폐업=0) 비율 : 43.7%


## 3-3. 점포수 기준 변경의 영향 ★

이전 파이프라인은 `일반_점포_수`(비프랜차이즈)로 5개 이상을 걸렀습니다. 총수 기준으로 바꾸면 어떤 칸이 새로 들어오는지 봅니다.

**들어오는 칸이 업종별로 편중돼 있다면**, 이전 분석의 표본 구성 자체가 업종별로 비대칭했다는 뜻입니다.

In [5]:
base = df[df["y"].notna() & df["총_유동인구_수"].notna() & (df["영역_면적"] > 0)]
old = base["일반_점포_수"] >= MIN_STORES     # 이전 기준
new = base["전체_점포_수"] >= MIN_STORES     # 현재 기준

print(f"이전 기준(일반_점포_수) 통과 : {old.sum():,}")
print(f"현재 기준(전체_점포_수) 통과 : {new.sum():,}   (+{new.sum()-old.sum():,}, "
      f"{(new.sum()-old.sum())/old.sum():+.1%})\n")

added = base[new & ~old]
t = (added.groupby("서비스_업종_코드_명")
     .agg(신규유입=("y", "size"), 순감소율=("y", "mean"))
     .join(base[old].groupby("서비스_업종_코드_명").size().rename("이전표본"))
     .assign(증가율=lambda x: (x["신규유입"]/x["이전표본"]))
     .sort_values("증가율", ascending=False))
display(t.round(3))
print("※ 증가율이 높은 업종 = 프랜차이즈 비중이 높아 이전에 과소 집계되던 업종")
print(f"※ 신규 유입 칸의 순감소율 {added['y'].mean():.1%} vs 기존 {base[old]['y'].mean():.1%}")

이전 기준(일반_점포_수) 통과 : 55,795
현재 기준(전체_점포_수) 통과 : 67,475   (+11,680, +20.9%)



,신규유입,순감소율,이전표본,증가율
서비스_업종_코드_명,,,,
치킨전문점,3332,0.189,854,3.902
패스트푸드점,2214,0.189,1687,1.312
제과점,1145,0.151,2988,0.383
분식전문점,1344,0.167,6226,0.216
커피-음료,1563,0.143,10483,0.149
중식음식점,420,0.152,3226,0.130
일식음식점,403,0.154,3293,0.122
양식음식점,345,0.180,3827,0.090
호프-간이주점,572,0.119,7876,0.073


※ 증가율이 높은 업종 = 프랜차이즈 비중이 높아 이전에 과소 집계되던 업종
※ 신규 유입 칸의 순감소율 16.9% vs 기존 27.3%


## 3-4. 매출 가용성 — 분석 표본 기준

전체 패널에서 매출 결측은 45% 지만, 그 대부분이 점포 2개 이하 칸의 비식별 처리입니다. **5개 이상 필터를 통과한 분석 표본에서는 상황이 다릅니다.**

In [6]:
d["매출_관측"] = d["당월_매출_금액"].notna()   # add_features 에서도 만들지만 여기서 먼저 쓴다
print(f"분석 표본 매출 관측률 {d['매출_관측'].mean():.1%}")
print(f"  골목상권만        {d[d['상권유형']=='골목상권']['매출_관측'].mean():.1%}\n")

display(d.groupby("서비스_업종_코드_명")
        .agg(관측률=("매출_관측", "mean"), 칸수=("매출_관측", "size"))
        .sort_values("관측률", ascending=False).round(3))
print("※ 관측률이 업종마다 다릅니다 → 매출을 매칭 공변량에 넣으면 대조군만 선택적으로 빠집니다.")
print("   보조질문 2(STEP 7)에서는 쓰되, PSM 주 분석에서는 제외하고 강건성 분석으로 다룹니다.")

분석 표본 매출 관측률 92.9%
  골목상권만        90.9%



,관측률,칸수
서비스_업종_코드_명,,
한식음식점,0.988,15677
호프-간이주점,0.986,8448
분식전문점,0.946,7570
중식음식점,0.938,3646
커피-음료,0.927,12046
치킨전문점,0.913,4186
일식음식점,0.873,3696
패스트푸드점,0.842,3901
제과점,0.839,4133


※ 관측률이 업종마다 다릅니다 → 매출을 매칭 공변량에 넣으면 대조군만 선택적으로 빠집니다.
   보조질문 2(STEP 7)에서는 쓰되, PSM 주 분석에서는 제외하고 강건성 분석으로 다룹니다.


## 3-5. 분기별 양성률

특정 분기만 튀면 계절성이거나 원본 문제입니다.

In [7]:
display(d.groupby("기준_년분기_코드")
        .agg(칸수=("y", "size"), 순감소율=("y", "mean"),
             평균점포=("전체_점포_수", "mean")).round(3))

,칸수,순감소율,평균점포
기준_년분기_코드,,,
20231,5706,0.229,21.067
20232,5731,0.261,21.101
20233,5701,0.223,21.168
20234,5730,0.236,21.198
20241,5706,0.254,21.398
20242,5678,0.276,21.488
20243,5622,0.262,21.547
20244,5586,0.284,21.612
20251,5534,0.261,21.558


## 3-6. 로그 변환

`영역_면적` 은 최소 1,854㎡ ~ 최대 2,462,734㎡ 로 **1,300배** 차이납니다. 점포수도 한쪽 꼬리가 깁니다. 원척도로 두면 극단값 몇 개가 평균과 매칭 거리 계산을 지배합니다.

### 점포 밀도에 대해

`점포밀도 = 전체_점포_수 / 영역_면적` 을 만들지만 **매칭 공변량으로 추가하지 않습니다.**

```
log(밀도) = log(점포수) − log(면적)
```

이미 매칭 중인 두 변수의 선형결합이라, 둘의 평균이 맞으면 로그 밀도의 평균도 자동으로 맞습니다. 별도 변수로 넣으면 완전 공선성이 됩니다. 해석·기술통계용으로만 둡니다.

In [8]:
def add_features(f):
    """로그 변환과 파생 변수. 학습 패널과 예측 프레임에 동일하게 적용한다."""
    f = f.copy()
    f["log_점포수"]   = np.log(f["전체_점포_수"])
    f["log_유동인구"] = np.log1p(f["총_유동인구_수"])
    f["log_면적"]     = np.log(f["영역_면적"])
    f["log_집객시설"] = np.log1p(f["집객시설_수"].fillna(0))
    # 파생 (매칭 공변량 아님)
    f["점포밀도"]   = f["전체_점포_수"] / f["영역_면적"] * 1000   # 1,000㎡당 점포수
    f["점포당매출"] = f["당월_매출_금액"] / f["전체_점포_수"]
    f["매출_관측"]  = f["당월_매출_금액"].notna()
    return f

d = add_features(d)
display(d[COVARIATES].describe().loc[["mean", "std", "min", "max"]].round(3))
print("[참고] 로그 밀도는 log_점포수 − log_면적 과 동일")
print(f"  상관계수 확인 : "
      f"{np.corrcoef(np.log(d['점포밀도']), d['log_점포수']-d['log_면적'])[0,1]:.6f}")

,log_점포수,log_유동인구,log_면적,log_집객시설
mean,2.584,13.657,11.634,3.187
std,0.849,1.016,0.842,0.995
min,1.609,2.197,7.525,0.000
max,6.522,15.969,14.717,6.389


[참고] 로그 밀도는 log_점포수 − log_면적 과 동일
  상관계수 확인 : 1.000000


### 공변량 상관

성향점수 모형에서 다중공선성이 문제될 수준인지 미리 봅니다.

In [9]:
display(d[COVARIATES + ["점포밀도", "외식_비중", "프랜차이즈_비율"]].corr().round(3))

,log_점포수,log_유동인구,log_면적,log_집객시설,점포밀도,외식_비중,프랜차이즈_비율
log_점포수,1.000,0.291,0.380,0.417,0.379,0.090,-0.131
log_유동인구,0.291,1.000,0.646,0.533,-0.273,-0.022,0.070
log_면적,0.380,0.646,1.000,0.725,-0.461,-0.235,0.150
log_집객시설,0.417,0.533,0.725,1.000,-0.245,-0.173,0.262
점포밀도,0.379,-0.273,-0.461,-0.245,1.000,0.278,-0.169
외식_비중,0.090,-0.022,-0.235,-0.173,0.278,1.000,-0.041
프랜차이즈_비율,-0.131,0.070,0.150,0.262,-0.169,-0.041,1.000


## 3-7. 예측용 채점 프레임 ★

`SCORE_Q`(20261) 행은 다음 분기가 없어 라벨을 만들 수 없습니다. 그래서 위 필터에서 탈락합니다 — **하지만 이 행들이 바로 예측 대상입니다.**

라벨만 없을 뿐 피처는 온전하므로, 학습 패널과 **동일한 필터·동일한 변환**을 적용해 따로 저장합니다. 여기서 변환이 어긋나면 STEP 9 예측이 조용히 틀립니다.

| 용도 | 피처 분기 | 타깃 분기 | 라벨 |
|---|---|---|---|
| 학습 | `TRAIN_QUARTERS` (20231~20253) | 20232~20254 | 있음 |
| 검증 | `VALID_Q` (20254) | 20261 | 있음 |
| 예측 | `SCORE_Q` (20261) | 20262 | 없음 |

> ⚠️ **학습에 `VALID_Q`(20254) 를 넣으면 20261 결과를 미리 보게 됩니다.** `config.TRAIN_QUARTERS` 가 이미 제외하고 있으니 STEP 8 에서는 반드시 그 상수를 쓰세요.

In [10]:
score = df[(df["기준_년분기_코드"] == SCORE_Q)
           & (df["전체_점포_수"] >= MIN_STORES)
           & (df["총_유동인구_수"].notna())
           & (df["영역_면적"] > 0)].copy()
score = add_features(score).drop(columns=["y", "다음분기idx", "순감소_현재"])

print(f"채점 프레임 (피처 {SCORE_Q} → 타깃 20262)  {score.shape}")
print(f"  라벨 컬럼 없음 확인 : {'y' not in score.columns}")

# 학습 패널과 컬럼이 일치하는지 — 여기서 어긋나면 STEP 9 가 조용히 틀린다
need = COVARIATES + ["점포밀도", "외식_비중", "프랜차이즈_비율", "매출_관측",
                     "자치구", "상권유형", "서비스_업종_코드_명"]
missing = [c for c in need if c not in score.columns]
print(f"  필수 컬럼 누락 : {missing or '없음'}")
assert not missing
print(f"  공변량 결측 : {dict(score[COVARIATES].isna().sum())}")

채점 프레임 (피처 20261 → 타깃 20262)  (5428, 35)
  라벨 컬럼 없음 확인 : True
  필수 컬럼 누락 : 없음
  공변량 결측 : {'log_점포수': np.int64(0), 'log_유동인구': np.int64(0), 'log_면적': np.int64(0), 'log_집객시설': np.int64(0)}


### 시간 분할 요약

In [11]:
split = pd.DataFrame([
    {"용도": "학습", "피처분기": f"{TRAIN_QUARTERS[0]}~{TRAIN_QUARTERS[-1]}",
     "행수": int(d["기준_년분기_코드"].isin(TRAIN_QUARTERS).sum()),
     "양성률": d.loc[d["기준_년분기_코드"].isin(TRAIN_QUARTERS), "y"].mean()},
    {"용도": "검증", "피처분기": str(VALID_Q),
     "행수": int((d["기준_년분기_코드"] == VALID_Q).sum()),
     "양성률": d.loc[d["기준_년분기_코드"] == VALID_Q, "y"].mean()},
    {"용도": "예측", "피처분기": str(SCORE_Q), "행수": len(score), "양성률": np.nan},
])
display(split.round(3))
assert VALID_Q not in TRAIN_QUARTERS, "검증 분기가 학습에 섞였습니다"
print("✅ 검증 분기가 학습에서 제외되어 있습니다")

,용도,피처분기,행수,양성률
0,학습,20231~20253,61996,0.254
1,검증,20254,5479,0.260
2,예측,20261,5428,NaN


✅ 검증 분기가 학습에서 제외되어 있습니다


## 3-8. 저장

In [12]:
d = d.drop(columns=["다음분기idx"])
d.to_pickle(PROC / "03_panel.pkl")
mb = (PROC / "03_panel.pkl").stat().st_size / 1024**2
print(f"[저장] 03_panel.pkl  {d.shape}  {mb:.1f} MB   ← 학습·검증")

score.to_pickle(PROC / "03_score.pkl")
mb2 = (PROC / "03_score.pkl").stat().st_size / 1024**2
print(f"[저장] 03_score.pkl  {score.shape}  {mb2:.1f} MB   ← STEP 9 예측 대상")
print(f"\n상권유형별 표본")
display(d.groupby("상권유형").agg(칸수=("y", "size"), 순감소율=("y", "mean")).round(3))

[저장] 03_panel.pkl  (67475, 37)  23.4 MB   ← 학습·검증
[저장] 03_score.pkl  (5428, 35)  1.8 MB   ← STEP 9 예측 대상

상권유형별 표본


,칸수,순감소율
상권유형,,
골목상권,34647,0.236
관광특구,696,0.378
발달상권,23011,0.292
전통시장,9121,0.224


---

## STEP 4 에서 할 일

- 상권유형별·자치구별 순감소율 훑기
- 새 변수 `프랜차이즈_비율`, `외식_비중`, `점포밀도` 의 분포와 y 와의 관계
- 관광특구는 상권 6개뿐이라 별도 결론을 내지 않습니다

STEP 8(예측)은 `03_panel.pkl` 을 `config.TRAIN_QUARTERS` / `VALID_Q` 로 나눠 쓰고, STEP 9 는 `03_score.pkl` 에 확률을 매깁니다.
